In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mit Pass Managern transpilieren

<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen die Verwendung dieser oder neuerer Versionen.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Der empfohlene Weg, einen Circuit zu transpilieren, besteht darin, einen gestuften Pass Manager zu erstellen und anschließend dessen `run`-Methode mit dem Circuit als Eingabe auszuführen. Diese Seite erklärt, wie du Quantum Circuits auf diese Weise transpilierst.
## Was ist ein (gestufter) Pass Manager?
Im Kontext des Qiskit SDK bezeichnet Transpilierung den Prozess der Umwandlung eines Eingabe-Circuits in eine Form, die für die Ausführung auf einem Quantengerät geeignet ist. Die Transpilierung erfolgt typischerweise in einer Abfolge von Schritten, die als Transpiler-Passes bezeichnet werden. Der Circuit wird von jedem Transpiler-Pass der Reihe nach verarbeitet, wobei die Ausgabe eines Passes zur Eingabe des nächsten wird. Ein Pass könnte beispielsweise den Circuit durchlaufen und alle aufeinanderfolgenden Sequenzen von Einzel-Qubit-Gates zusammenführen, und der nächste Pass könnte diese Gates dann in den Basissatz des Zielgeräts synthetisieren. Die in Qiskit enthaltenen Transpiler-Passes befinden sich im Modul [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

Ein Pass Manager ist ein Objekt, das eine Liste von Transpiler-Passes speichert und diese auf einem Circuit ausführen kann. Erstelle einen Pass Manager, indem du einen [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) mit einer Liste von Transpiler-Passes initialisierst. Um die Transpilierung auf einem Circuit auszuführen, rufe die [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run)-Methode mit einem Circuit als Eingabe auf.

Ein gestufter Pass Manager ist eine besondere Art von Pass Manager, der eine Abstraktionsebene über einem normalen Pass Manager darstellt. Während ein normaler Pass Manager aus mehreren Transpiler-Passes besteht, setzt sich ein gestufter Pass Manager aus mehreren *Pass Managern* zusammen. Dies ist eine nützliche Abstraktion, da die Transpilierung typischerweise in diskreten Stufen erfolgt, wie in [Transpiler-Stufen](transpiler-stages) beschrieben, wobei jede Stufe durch einen Pass Manager repräsentiert wird. Gestufte Pass Manager werden durch die Klasse [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) dargestellt. Der Rest dieser Seite beschreibt, wie du (gestufte) Pass Manager erstellst und anpasst.
## Einen voreingestellten gestuften Pass Manager generieren
Um einen voreingestellten gestuften Pass Manager mit sinnvollen Standardwerten zu erstellen, verwende die Funktion [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager):

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Um einen Circuit oder eine Liste von Circuits mit einem Pass Manager zu transpilieren, übergib den Circuit oder die Liste von Circuits an die `run`-Methode. Lass uns das an einem Zwei-Qubit-Circuit ausprobieren, der aus einem Hadamard gefolgt von zwei benachbarten CX-Gates besteht:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

Unter [Transpilierungsstandards und Konfigurationsoptionen](defaults-and-configuration-options) findest du eine Beschreibung der möglichen Argumente für die Funktion `generate_preset_pass_manager`. Die Argumente von `generate_preset_pass_manager` entsprechen den Argumenten der Funktion [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="Hast du Probleme, dir Pass-Manager-Details zu merken? Frag den Qiskit Code Assistant." />

Wenn die voreingestellten Pass Manager deine Anforderungen nicht erfüllen, passe die Transpilierung an, indem du (gestufte) Pass Manager oder sogar Transpilierungs-Passes selbst erstellst. Der Rest dieser Seite beschreibt, wie du Pass Manager erstellst. Anleitungen zum Erstellen von Transpilierungs-Passes findest du unter [Eigenen Transpiler-Pass schreiben](custom-transpiler-pass).

## Eigenen Pass Manager erstellen

Das Modul [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) enthält viele Transpiler-Passes, die zur Erstellung von Pass Managern verwendet werden können. Um einen Pass Manager zu erstellen, initialisiere einen `PassManager` mit einer Liste von Passes. Der folgende Code erstellt beispielsweise einen Transpiler-Pass, der benachbarte Zwei-Qubit-Gates zusammenführt und sie dann in eine Basis aus [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate)-, [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate)- und [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate)-Gates synthetisiert.

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

Um diesen Pass Manager in Aktion zu demonstrieren, teste ihn an einem Zwei-Qubit-Circuit, der aus einem Hadamard gefolgt von zwei benachbarten CX-Gates besteht:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

Um den Pass Manager auf dem Circuit auszuführen, rufe die Methode `run` auf.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

Ein fortgeschritteneres Beispiel, das zeigt, wie du einen Pass Manager für die Fehlersuppression durch dynamisches Entkoppeln erstellst, findest du unter [Einen Pass Manager für dynamisches Entkoppeln erstellen](dynamical-decoupling-pass-manager).

## Einen gestuften Pass Manager erstellen

Ein `StagedPassManager` ist ein Pass Manager, der aus einzelnen Stufen besteht, wobei jede Stufe durch eine `PassManager`-Instanz definiert wird. Du kannst einen `StagedPassManager` erstellen, indem du die gewünschten Stufen angibst. Der folgende Code erstellt beispielsweise einen gestuften Pass Manager mit zwei Stufen, `init` und `translation`. Die Stufe `translation` wird durch den zuvor erstellten Pass Manager definiert.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

Die Anzahl der Stufen, die du in einen gestuften Pass Manager einfügen kannst, ist unbegrenzt.

Eine weitere nützliche Möglichkeit, einen gestuften Pass Manager zu erstellen, besteht darin, mit einem voreingestellten gestuften Pass Manager zu beginnen und dann einige der Stufen auszutauschen. Der folgende Code generiert beispielsweise einen voreingestellten Pass Manager mit Optimierungsstufe 3 und gibt dann eine benutzerdefinierte `pre_layout`-Stufe an.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

Die [Stufengeneratorfunktionen](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) können beim Aufbau benutzerdefinierter Pass Manager hilfreich sein.
Sie erzeugen Stufen, die häufig verwendete Funktionalitäten in vielen Pass Managern bereitstellen.
Beispielsweise kann [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) verwendet werden, um eine Stufe zu generieren,
die ein ausgewähltes initiales `Layout` aus einem Layout-Pass auf das angegebene Zielgerät „einbettet".

## Nächste Schritte
> **Tip:** - [Einen eigenen Transpiler-Pass schreiben](custom-transpiler-pass).
>     - [Einen Pass Manager für dynamisches Entkoppeln erstellen](dynamical-decoupling-pass-manager).
>     - Um mehr über die Funktion `generate_preset_passmanager` zu erfahren, lies das Thema [Transpilierungs-Standardeinstellungen und Konfigurationsoptionen](defaults-and-configuration-options).
>     - Probiere die Anleitung [Transpiler-Einstellungen vergleichen](/guides/circuit-transpilation-settings) aus.
>     - Lies die [Transpiler-API-Dokumentation.](https://docs.quantum.ibm.com/api/qiskit/transpiler)